# Analyze synchronization for pitching URANS 2D

In [27]:
import matplotlib.pyplot as plt

from glob import glob
from os import makedirs
from os.path import join, exists
from torch import where, tensor, from_numpy, cat

from utils import load_force_coeffs, interpolate_uniform, compute_fft

In [28]:
# validation @ Ma = 0.73
u_inf = 242.16629
alpha0 = 3.5

# chord length
chord = 1

# Max. amplitude at f = 14.27309 Hz (Sr = 0.05894) for no pitching case (avg. over 339 CTU)
f_b = 14.27309

# start of quasi-steady state -> chosen based on course of cl
tstar_start = 145

# use latex fonts
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360})

In [10]:
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6_pitching")
save_dir = join("..", "run", "plots", "URANS_pitching", "URANS_blockMesh", "first_test_3.5deg")

# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

# get all available amplitudes, since we do pitching frequency sweeps for a constant pitching amplitude
amplitudes = {a.split("A")[-1].split("_")[0]: {} for a in glob(join(load_dir, "A*"))}

In [11]:
def interpolate_cl(load_path: str):
    # load the cl values
    _cl = load_force_coeffs(load_path, usecols=[0, 4], names=["t", "cy"])

     # re-sample the force coefficients since the simulation is run with a variable time step
    try:
        idx_start = where(abs(tensor(_cl["t"].values) * u_inf / chord - tstar_start) < 1e-4)[0][0].item()

        # remove the transient phase and duplicates (resulting from dt < write precision)
        _cl.drop(list(range(0, idx_start)), inplace=True)
        _cl.reset_index(inplace=True, drop=True)

        # interpolate values
        _t_uniform, _cy_inter = interpolate_uniform(_cl["t"].values, _cl["cy"].values)

        return _t_uniform, _cy_inter
    except IndexError:
        print("Could not find a valid start index.")

In [ ]:
# number of frequencies to keep
n_freq = 100
for a in amplitudes.keys():
    print(f"Loading all available pitching frequencies for pitching amplitude {a} deg.")

    # get all available frequencies
    freq, all_ffts = [], []
    for f in sorted(glob(join(load_dir, f"A{a}_f*")), key=lambda x: float(x.split("f")[-1])):
        # save the pitching frequencies tested
        freq.append(float(f.split("_")[-1].strip("f")))

        # load cl only, strip the initial transient and interpolate it to equidistant spacing
        t_uniform, cl_inter = interpolate_cl(f)

        # perform FFT on the cl
        _f, _a = compute_fft(cl_inter, (t_uniform[1] - t_uniform[0]).item())

        # convert to tensor for easier access, only keep the first few frequencies
        all_ffts.append(cat([from_numpy(_f).unsqueeze(-1), from_numpy(_a).unsqueeze(-1)], dim=-1)[:n_freq, :].unsqueeze(0))

    # convert to tensor for easier access
    all_ffts = cat(all_ffts, dim=0)

    # add pitching frequencies and FFT results to corresponding pitching amplitude
    amplitudes[a]["f_p"] = tensor(freq)
    amplitudes[a]["f_cl"] = all_ffts[:, :, 0]
    amplitudes[a]["a_cl"] = all_ffts[:, :, 1]

# free-up some memory
del t_uniform, cl_inter, _f, _a, all_ffts

In [ ]:
# plot results of FFT's -> on x-axis fp/fb, on y-axis the response in cl and the color the amplitudes
for a in amplitudes.keys():
    fig, ax = plt.subplots(figsize=(6, 3))

    # we have to expand the frequencies tested to match the dimensions of the FFTs (N_frequencies_pitching, N_frequencies_cl, 2) -> 2 = [f, a]
    freqs_b = amplitudes[a]["f_p"].unsqueeze(-1).repeat(1, amplitudes[a]["f_cl"].shape[1]) / f_b

    # plot
    print(amplitudes[a]["a_cl"].min().item(), amplitudes[a]["a_cl"].max().item())
    ax.contourf(freqs_b, amplitudes[a]["f_cl"] / f_b, amplitudes[a]["a_cl"], cmap="plasma", levels=100, extend="both", vmin=1e-8, vmax=1e-2)

    # mark fp / f_b = 1
    ax.axhline(y=1, color="red", linestyle=":")
    ax.axvline(x=1, color="red", linestyle=":")

    ax.set_xlabel(r"$f_p \, / \, f_b \quad [-]$")
    ax.set_ylabel(r"$f_{c_L} \, / \, f_b \quad [-]$")
    ax.set_ylim(0.25, 2)
    # ax.set_xlim(0.3, 2)
    fig.tight_layout()
    fig.subplots_adjust()
    plt.savefig(join(save_dir, f"FFTs_vs_pitching_frequency_A{a}.png"))
    plt.show()

del freqs_b

In [ ]:
# for each frequency-amplitude combination check, if we have amplitude > threshold for both pitching and buffet frequency
# if yes -> no synchronization else synchronization
idx_sync = {}
for a in amplitudes.keys():
    idx_sync[a] = []
    for sim in range(amplitudes[a]["f_cl"].size(0)):
        # amplitude of the buffet frequency
        _idx_buffet = where((amplitudes[a]["f_cl"][sim, :] - f_b).abs() < 1e-5)
        a0 = amplitudes[a]["a_cl"][sim, _idx_buffet]

        # get the index of the pitching frequency in the spectrum
        _idx_pitching = (amplitudes[a]["f_cl"][sim, :] - amplitudes[a]["f_p"][sim]).abs().argmin()

        # check some infos
        print("{:.3f}".format(amplitudes[a]["f_cl"][sim, _idx_buffet].item()), "{:.3f}".format(amplitudes[a]["f_cl"][sim, _idx_pitching].item()),
              "{:.2e}".format(amplitudes[a]["a_cl"][sim, _idx_pitching].item()), amplitudes[a]["f_p"][sim].item())

        # TODO: this threshold is somewhat arbitrary for now
        if amplitudes[a]["a_cl"][sim, _idx_pitching] > 0.5 * a0:    #  and (amplitudes[a]["f_cl"][sim, _idx_pitching] - amplitudes[a]["f_p"][sim]).abs() < 0.5:
            idx_sync[a].append(sim)

print(idx_sync)

In [ ]:
# then plot, select marker to visualize Arnold's tongues -> fp / fb vs. delta alpha
fig, ax = plt.subplots(figsize=(6, 3))

for a in amplitudes.keys():
    no_sync = [i for i in range(amplitudes[a]["f_p"].size(0)) if i not in idx_sync[a]]

    if a == list(amplitudes.keys())[0]:
        ax.scatter(amplitudes[a]["f_p"][idx_sync[a]] / f_b, [float(a)/alpha0] * len(idx_sync[a]), marker="o", color="C0", label=r"$\mathrm{synchronization}$")
        ax.scatter(amplitudes[a]["f_p"][no_sync] / f_b, [float(a)/alpha0] * len(no_sync), marker="x", color="red", label=r"$\mathrm{no~synchronization}$")
    else:
        ax.scatter(amplitudes[a]["f_p"][idx_sync[a]] / f_b, [float(a)/alpha0] * len(idx_sync[a]), marker="o", color="C0")
        ax.scatter(amplitudes[a]["f_p"][no_sync] / f_b, [float(a)/alpha0] * len(no_sync), marker="x", color="red")

ax.set_xlabel(r"$f_p \, / \, f_b \quad [-]$")
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_ylabel(r"$\Delta \alpha_\mathrm{inlet} \, / \, \alpha_0 \quad [-]$")
fig.tight_layout()
fig.legend(loc="upper center", ncols=2)
fig.subplots_adjust(top=0.86)
plt.savefig(join(save_dir, "Arnold_analysis.png"))
plt.show()